In [2]:
import pandas as pd
import duckdb


In [4]:
con = duckdb.connect('credit_risk.db')

with open('C:/Users/kweec/python_files/risk-project/sql/01_raw.sql', 'r') as f:
    sql = f.read()
con.execute(sql)


# Проверяем
result = con.execute("SELECT COUNT(*) FROM raw.loans").fetchone()
print(f"Строк загружено: {result[0]}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Строк загружено: 2260668


In [5]:
con.execute("DESCRIBE raw.loans").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,member_id,VARCHAR,YES,None,None,None
2,loan_amnt,DOUBLE,YES,None,None,None
3,funded_amnt,DOUBLE,YES,None,None,None
4,funded_amnt_inv,DOUBLE,YES,None,None,None
...,...,...,...,...,...,...
146,settlement_status,VARCHAR,YES,None,None,None
147,settlement_date,VARCHAR,YES,None,None,None
148,settlement_amount,DOUBLE,YES,None,None,None
149,settlement_percentage,DOUBLE,YES,None,None,None


In [6]:
with open('C:/Users/kweec/python_files/risk-project/sql/02_staging.sql', 'r') as f:
    sql = f.read()
con.execute(sql)

In [7]:
df = con.execute("""
    SELECT 
        is_default,
        COUNT(*) as cnt,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 1) as pct
    FROM staging.loans
    GROUP BY is_default
""").df()
print(df)

   is_default      cnt   pct
0           0  1073336  80.0
1           1   267774  20.0


In [8]:
with open('C:/Users/kweec/python_files/risk-project/sql/03_mart.sql', 'r') as f:
    sql = f.read()
con.execute(sql)

# проверка
con.execute("SELECT * FROM mart.features LIMIT 5").df()

con.close()